# PID lifetime model of Hacke et al. for PVDeg

PVDeg 0.7.5 has no potential-induced degradation (PID) function. The companion notebook `pid_oh2017.ipynb` adds the
voltage-Peck rate law of Oh et al. This notebook adds the second published PID model, the lifetime model of Hacke et
al., and checks each of its numbers against the paper.

**Source.** P. Hacke, R. Smith, K. Terwilliger, S. Glick, D. Jordan, S. Johnston, M. Kempe, S. Kurtz, "Testing and
analysis for lifetime prediction of crystalline silicon PV modules undergoing degradation by system voltage stress",
*IEEE Journal of Photovoltaics* 3(1), 246–253 (2013). doi:10.1109/JPHOTOV.2012.2222351.
The text checked here is the open NREL preprint of the same paper, NREL/CP-5200-54109 (July 2012).

The paper held one module design at 85 % RH and −600 V in a chamber at 50, 60 and 85 °C, and two replicas outdoors in
Florida under −600 V. It reports five things that can be checked: the shunt-resistance power estimate (Table I),
the activation energy of the failure rate (Fig. 3), the 25 °C lifetime (Fig. 3), the Florida power curve (Fig. 4) and
the acceleration factors (Fig. 5). Values read from figures were digitized from the preprint figures and are marked
as such.

## The model

Power falls linearly with the square of time (Section III.C and III.D):

$$\frac{P(t)}{P_\mathrm{max,0}} = P_0 - k\,t^2$$

Failure times at a fixed criterion follow a lognormal distribution with one shape factor; the failure rate
$r = 1/t_\mathrm{fail}$ follows Arrhenius (Fig. 3):

$$r(T) = r_\mathrm{ref}\,\exp\!\left[-\frac{E_a}{k_B}\left(\frac{1}{T}-\frac{1}{T_\mathrm{ref}}\right)\right]$$

The acceleration factor is the field lifetime over the chamber lifetime at the same failure level (Section III.D):

$$AF(T) = \frac{t_\mathrm{field}}{t_\mathrm{chamber}(T)}$$

Fraction of power lost to a shunt, from Fahrenbruch and Bube (Section II):

$$\frac{\Delta P}{P_\mathrm{max,0}} = \frac{V_\mathrm{oc}}{J_\mathrm{sc}\,R_\mathrm{sh}}$$

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

K_B = 8.617333e-5   # Boltzmann constant, eV/K


def power_t2(t_h, p0, k):
    # normalized power after t_h hours: linear in time squared
    return p0 - k * np.asarray(t_h, dtype=float) ** 2


def time_to_level(level, p0, k):
    # hours until the normalized power reaches `level`
    return np.sqrt((p0 - level) / k)


def shunt_power_loss(voc, jsc, rsh):
    # fraction of power lost to the shunt (Fahrenbruch and Bube)
    return voc / (jsc * rsh)


def arrhenius_fit(temp_c, y):
    # least-squares line of ln(y) against 1/T; returns Ea (eV), its standard error and the line
    x = 1 / (np.asarray(temp_c) + 273.15)
    (slope, icpt), cov = np.polyfit(x, np.log(y), 1, cov="unscaled")
    resid = np.log(y) - (slope * x + icpt)
    s2 = resid @ resid / max(len(x) - 2, 1)
    return -slope * K_B, np.sqrt(cov[0, 0] * s2) * K_B, (slope, icpt)


def arrhenius_eval(line, temp_c):
    slope, icpt = line
    return np.exp(slope / (np.asarray(temp_c) + 273.15) + icpt)


rows = []   # summary: quantity, paper, ours


def compare(name, paper, ours):
    rows.append([name, paper, ours, 100 * (ours - paper) / paper])
    print(f"{name}: paper {paper:.4g}, ours {ours:.4g}, difference {100 * (ours - paper) / paper:+.1f} %")

## Check 1: the photocurrent used for superposition

The paper translates the dark curves by 7.3 A and says this is 30 mA/cm² for 15.6 cm × 15.6 cm cells:

$$J_\mathrm{sc} = \frac{I}{A} = \frac{7.3\ \mathrm{A}}{(15.6\ \mathrm{cm})^2}$$

In [2]:
jsc = 7.3 / 15.6 ** 2 * 1000      # mA/cm2
compare("Jsc from 7.3 A (mA/cm2)", 30.0, jsc)

Jsc from 7.3 A (mA/cm2): paper 30, ours 30, difference -0.0 %


## Check 2: Table I, power loss from shunt resistance

Table I gives two shunt-resistance estimates with their power: 333 Ω·cm² → 0.942 and 660 Ω·cm² → 0.971.
The loss is $V_\mathrm{oc}/(J_\mathrm{sc}R_\mathrm{sh})$, so one module constant $V_\mathrm{oc}/J_\mathrm{sc}$ fixes both.
We take it from the first column and predict the second. The implied $V_\mathrm{oc}$ (with $J_\mathrm{sc}$ = 30 mA/cm²)
is a cross-check that the constant is physical; the paper does not print $V_\mathrm{oc}$.

$$\frac{V_\mathrm{oc}}{J_\mathrm{sc}} = (1-0.942)\times 333,\qquad P(660) = 1 - \frac{V_\mathrm{oc}/J_\mathrm{sc}}{660}$$

In [3]:
voc_over_jsc = (1 - 0.942) * 333                     # ohm cm2, fitted on the 333 column
print(f"Voc/Jsc = {voc_over_jsc:.1f} ohm cm2 -> implied Voc = {voc_over_jsc * 0.030:.3f} V per cell")
compare("Table I power at Rsh 660 ohm cm2", 0.971, 1 - shunt_power_loss(voc_over_jsc * 0.030, 0.030, 660))
print("measured by solar simulator: 0.765, so the shunt alone misses most of the loss, as the paper says")

Voc/Jsc = 19.3 ohm cm2 -> implied Voc = 0.579 V per cell
Table I power at Rsh 660 ohm cm2: paper 0.971, ours 0.9707, difference -0.0 %
measured by solar simulator: 0.765, so the shunt alone misses most of the loss, as the paper says


## Check 3: activation energy of the failure rate (Fig. 3 inset)

The inset of Fig. 3 plots the failure rate (h⁻¹) against 1000/T for the three chamber temperatures.
The three points were digitized from the figure. The paper reports $E_a$ = 0.726 ± 0.053 eV.

$$\ln r = \ln r_0 - \frac{E_a}{k_B}\,\frac{1}{T}$$

In [4]:
fig3 = pd.DataFrame({"T (C)": [85, 60, 50],
                     "failure rate (1/h), digitized": [0.1980, 0.03376, 0.01549]})
ea, ea_se, line = arrhenius_fit(fig3["T (C)"], fig3["failure rate (1/h), digitized"])
print(fig3.to_string(index=False))
print(f"Ea = {ea:.3f} +/- {ea_se:.3f} eV (fit standard error)")
compare("Ea of failure rate, Fig. 3 (eV)", 0.726, ea)

 T (C)  failure rate (1/h), digitized
    85                        0.19800
    60                        0.03376
    50                        0.01549
Ea = 0.726 +/- 0.001 eV (fit standard error)
Ea of failure rate, Fig. 3 (eV): paper 0.726, ours 0.7263, difference +0.0 %


## Check 4: the 25 °C lifetime (Fig. 3)

The paper extrapolates the failure distribution to 25 °C, 85 % RH, −600 V and says its peak is around 550 h.
The median of a lognormal is its peak on the log-time axis of Fig. 3, so we compare $1/r(25\,°\mathrm{C})$.

In [5]:
t25 = 1 / arrhenius_eval(line, 25.0)
compare("time to failure at 25 C (h)", 550, float(t25))

time to failure at 25 C (h): paper 550, ours 576.1, difference +4.7 %


## Check 5: the Florida power curve (Fig. 4)

The line in Fig. 4 was digitized along 379 pixel columns and converted to data units with the axis ticks
(the time axis of Fig. 4 is linear in $t^2$). That gives $P_0$ and $k$ of the paper's fitted line. The text says the
modules reached 0.89 after about 10 months, and the line extrapolates to 0.8 at about 12,000 h (1.37 y).

$$t(P) = \sqrt{\frac{P_0 - P}{k}}$$

In [6]:
P0_FIELD, K_FIELD = 0.99541, 1.8723e-9      # digitized from the Fig. 4 line, k in 1/h^2
t089 = time_to_level(0.89, P0_FIELD, K_FIELD)
t080 = time_to_level(0.80, P0_FIELD, K_FIELD)
t095 = time_to_level(0.95, P0_FIELD, K_FIELD)
print(f"line reaches 0.89 at {t089:.0f} h = {t089 / 730.5:.1f} months")
compare("months to 0.89 in Florida", 10, t089 / 730.5)
compare("hours to 0.80 in Florida", 12000, t080)
print(f"line reaches 0.95 (the failure level for Fig. 5) at {t095:.0f} h")

line reaches 0.89 at 7503 h = 10.3 months
months to 0.89 in Florida: paper 10, ours 10.27, difference +2.7 %
hours to 0.80 in Florida: paper 1.2e+04, ours 1.022e+04, difference -14.9 %
line reaches 0.95 (the failure level for Fig. 5) at 4925 h


The line drawn in Fig. 4 crosses 0.80 inside the plotted range, near 10,200 h, while the text says about
12,000 h. The figure and the text of the paper disagree with each other here; the 12,000/8760 = 1.37 y arithmetic in
the text is itself exact. Both numbers are kept in the summary.

## Check 6: the acceleration factors (Fig. 5)

Fig. 5 is a vector figure, so the three acceleration factors were read from the drawing commands of the PDF.
If the failure rate is Arrhenius, $\ln AF$ is linear in $1/T$ with the same $E_a$ as Fig. 3:

$$\ln AF(T) = \frac{E_a}{k_B}\,\frac{1}{T} + c$$

Dividing the Florida time to 0.95 from Check 5 by each $AF$ also gives the chamber time to 0.95.

In [7]:
fig5 = pd.DataFrame({"T (C)": [84.948, 59.952, 49.946],
                     "AF, read from vector figure": [1945.6, 430.5, 124.7]})
ea_af, ea_af_se, _ = arrhenius_fit(fig5["T (C)"], fig5["AF, read from vector figure"])
fig5["chamber hours to 0.95"] = t095 / fig5["AF, read from vector figure"]
print(fig5.round(2).to_string(index=False))
print(f"Ea from AF = {ea_af:.3f} +/- {ea_af_se:.3f} eV; paper range 0.726 +/- 0.053 = 0.673 to 0.779 eV")
compare("Ea from acceleration factors, Fig. 5 (eV)", 0.726, ea_af)

 T (C)  AF, read from vector figure  chamber hours to 0.95
 84.95                       1945.6                   2.53
 59.95                        430.5                  11.44
 49.95                        124.7                  39.49
Ea from AF = 0.755 +/- 0.124 eV; paper range 0.726 +/- 0.053 = 0.673 to 0.779 eV
Ea from acceleration factors, Fig. 5 (eV): paper 0.726, ours 0.7551, difference +4.0 %


## Summary

Each row is one number printed in the paper and the same number from this notebook.

In [8]:
summary = pd.DataFrame(rows, columns=["quantity", "paper", "this notebook", "difference (%)"])
summary["agreement (%)"] = 100 - summary["difference (%)"].abs()
print(summary.round(3).to_string(index=False))

                                 quantity     paper  this notebook  difference (%)  agreement (%)
                  Jsc from 7.3 A (mA/cm2)    30.000         29.997          -0.011         99.989
         Table I power at Rsh 660 ohm cm2     0.971          0.971          -0.027         99.973
          Ea of failure rate, Fig. 3 (eV)     0.726          0.726           0.046         99.954
              time to failure at 25 C (h)   550.000        576.055           4.737         95.263
                months to 0.89 in Florida    10.000         10.271           2.715         97.285
                 hours to 0.80 in Florida 12000.000      10216.113         -14.866         85.134
Ea from acceleration factors, Fig. 5 (eV)     0.726          0.755           4.005         95.995


Six of the seven numbers agree within 5 %, three of them within 0.1 %. The Fig. 5 activation energy lies inside the paper's own
uncertainty of ±0.053 eV. The one miss is the 12,000 h warranty failure in the text: the line in Fig. 4 itself reaches
0.80 near 10,200 h, so the paper is not self-consistent at that point, and no change to the model can match both.